# 04 - Polling coverage

Each poll returns the next `limit` departures of a station. If those departures cover **less time than the interval until the next poll**, departures between the two polls are never captured.

This notebook measures, for one ingestion batch, how many minutes of departures each station covers.

**Rule:** `minutes_covered` must exceed the polling interval, with a margin for rush hours.

In [ ]:
import pandas as pd

BRONZE = "/workspace/data/bronze/transport_stationboard"

df = pd.read_parquet(BRONZE)
batches = sorted(df["_batch_id"].unique())
print("Available batches:", batches)

## Coverage of one batch
By default the latest batch. Set `BATCH` to another value from the list above to compare.

In [ ]:
BATCH = batches[-1]

coverage = (
    df[df["_batch_id"] == BATCH]
    .groupby("station_id")["departure_planned_utc"]
    .agg(first="min", last="max", departures="count")
)
coverage["minutes_covered"] = (coverage["last"] - coverage["first"]).dt.total_seconds() / 60
print("Batch:", BATCH)
coverage.sort_values("minutes_covered")

## Results

| Batch | limit | Lowest coverage | Note |
|---|---|---|---|
| 20261018T14…Z (Sunday afternoon) | 30 | Zürich HB 26 min, Bern 30 min | below 60 min: hourly polling loses departures |
| 20261018T144846Z (Sunday afternoon) | 100 | Zürich HB 82 min, Bern 112 min | all stations > 80 min: about 5× the 15-min polling interval |

**Decision:** poll every **15 minutes** with `limit = 100`. To be re-measured on a weekday rush hour (07:00-08:00 local).